# temporal_event_model v1 checkpoint prediction inspection

Set `CHECKPOINT` to a trained temporal checkpoint. The notebook can also run with the dummy model path for shape checks.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in Path.cwd().resolve().parents if (p / 'research').exists()), Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CHECKPOINT = Path('')
ROOT

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from research.temporal_event_model.v1.config import ModelConfig
from research.temporal_event_model.v1.model import TemporalEventPredictor

class DummyEventEncoder(nn.Module):
    def __init__(self, embedding_dim=32):
        super().__init__()
        self.projection = nn.Linear(14 + 128 * 16, embedding_dim)
    def forward(self, header_uint8, events_uint8):
        x = torch.cat([header_uint8.float(), events_uint8.float().flatten(1)], dim=1) / 255.0
        return self.projection(x)

B, K, H, E = 4, 16, 1, 32
model = TemporalEventPredictor(
    event_encoder=DummyEventEncoder(E),
    config=ModelConfig(embedding_dim=E, temporal_d_model=64, temporal_layers=1, temporal_heads=4, decoder_layers=1),
    context_chunks=K,
    target_chunks=H,
)
if CHECKPOINT:
    payload = torch.load(CHECKPOINT, map_location='cpu')
    model.load_state_dict(payload['model_state_dict'], strict=False)
model.eval()
context_header = torch.randint(0, 256, (B, K, 14), dtype=torch.uint8)
context_events = torch.randint(0, 256, (B, K, 128, 16), dtype=torch.uint8)
with torch.no_grad():
    output = model(context_header, context_events)
    probs = torch.sigmoid(output.event_bit_logits).flatten(1)
plt.figure(figsize=(12, 4))
for row in range(min(B, 4)):
    plt.plot(probs[row, :512].numpy(), label=f'sample {row}')
plt.title('First 512 future event-bit probabilities')
plt.legend()
plt.show()